In [ ]:
from IPython.display import Markdown, display
import os
from dotenv import load_dotenv
from pathlib import Path
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.generation.generator import generate_answer
from src.graph.rag_graph import rag_pipeline

In [ ]:
load_dotenv()
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent

In [ ]:
question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)
chunks = answer["source_chunks"]
avg_score = sum(c["score"] for c in chunks) / len(chunks)
display(Markdown(answer["answer"]))
print(f"Model: {answer['model']} | Tokens: {answer['token_count']} | Avg_score: {avg_score}")

In [ ]:
question = "how are you?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)
chunks = answer["source_chunks"]
avg_score = sum(c["score"] for c in chunks) / len(chunks)
display(Markdown(answer["answer"]))
print(f"Model: {answer['model']} | Tokens: {answer['token_count']} | Avg_score: {avg_score}")

In [ ]:
question = "hello how are you?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)
chunks = answer["source_chunks"]
avg_score = sum(c["score"] for c in chunks) / len(chunks)
display(Markdown(answer["answer"]))
print(f"Avg_score: {avg_score}")

In [ ]:
def check_quality(reranked_chunks):
    if not reranked_chunks:
        return False
    avg_score = sum(c["score"] for c in reranked_chunks) / len(reranked_chunks)
    return avg_score > 0

In [ ]:
check_quality(answer["source_chunks"])

In [ ]:
from src.generation.llm_router import generate

def rewrite_query(question: str) -> str:
    prompt = f"""Rephrase this question to be more specific for searching academic research papers about AI and machine learning.
Keep the same intent but use more precise technical terms.
    
Original question: {question}

Rewritten question:"""
    return generate(prompt, model="openai")

In [ ]:
rewrite_query("what are the key concepts in the paper?")

In [ ]:
max_retries = 2
retries = 0

results = retrieve(question)
reranked = rerank(question, results, top_n=5)

while not check_quality(reranked) and retries < max_retries:
    print(f"Low quality — rewriting query (attempt {retries + 1})")
    question = rewrite_query(question)
    print(f"Rewritten: {question}")
    results = retrieve(question)
    reranked = rerank(question, results, top_n=5)
    retries += 1

if check_quality(reranked):
    answer = generate_answer(question, reranked)
    display(Markdown(answer["answer"]))
else:
    print("Could not find relevant information after retrying.")
    display(Markdown(answer["answer"]))

In [ ]:
def run_rag_pipeline(question: str):
    max_retries = 2
    retries = 0

    results = retrieve(question)
    reranked = rerank(question, results, top_n=5)

    while not check_quality(reranked) and retries < max_retries:
        print(f"Low quality — rewriting query (attempt {retries + 1})")
        question = rewrite_query(question)
        print(f"Rewritten: {question}")
        results = retrieve(question)
        reranked = rerank(question, results, top_n=5)
        retries += 1

    if check_quality(reranked):
        answer = generate_answer(question, reranked)
    else:
        answer = generate_answer(question, reranked)
        print("Could not find relevant information after retrying.")

    display(Markdown(answer["answer"]))
    return answer

In [ ]:
run_rag_pipeline("who are you?")

In [ ]:
run_rag_pipeline("what is the main idea")

In [ ]:
run_rag_pipeline("explain me about Api")

In [ ]:
run_rag_pipeline("what is attention in the real world and in this papers?")

In [ ]:
run_rag_pipeline("what is attention")

In [ ]:
from src.graph.rag_graph import rag_pipeline

result = rag_pipeline.invoke({
    "question": "What is the attention mechanism?",
    "original_question": "What is the attention mechanism?",
    "chunks": [],
    "reranked": [],
    "answer": {},
    "retries": 0,
    "quality_passed": False,
})

print(result["answer"]["answer"])

In [ ]:
result = rag_pipeline.invoke({
    "question": "hello how are you?",
    "original_question": "hello how are you?",
    "chunks": [],
    "reranked": [],
    "answer": {},
    "retries": 0,
    "quality_passed": False,
})

if result.get("error"):
    display(Markdown(result["error"]))
else:
    display(Markdown(result["answer"]["answer"]))


In [ ]:
question = "what is AI"
prompt = (
    "Is this question related to artificial intelligence, machine learning, or academic research topics? "
    "Reply with only 'yes' or 'no'.\n\n"
    f"Question: {question}"
)
response = generate(prompt, model="openai")
print(f"Response: '{response}'")